# 选择一个资料没有找全的问题

先不选改法，只保存一个可以重复运行的问题和原始检索结果。

这里选择一道需要连续阅读第 41～44 页的 LDA 推导题。基础检索同样返回 4 页，却只找到了第 44 页。预期页码只在结果出来后用于核对，不会传给检索器。

In [1]:
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

In [2]:
from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages, page_coverage
from common.nontraining_utils import load_annotation

case_data = load_query_catalog()
case = next(item for item in case_data if item["id"] == "lda_recursive_derivation")
search = build_bm25_search(load_pdf_pages())
query = case["query"]
print("问题：", query)

问题： LDA 从投影分离目标怎样推到 N−1 个最大广义特征值及其特征向量？请给出中间优化关系。


## 保存原始结果

使用 BM25 作为容易复现的基础方案，取前 4 页。此处只确认资料是否找全，不在这一节尝试修复。

In [3]:
# 先检索原问题；必要页清单只在检索完成后用于核对。
baseline = search(query, top_k=4)
annotation = load_annotation(case["id"])
expected_pages = annotation["expected_pages"]
hit_count, coverage, hit_pages = page_coverage(expected_pages, baseline)

print("返回页面：", [item.page for item in baseline])
print("检索后用于核对的必要页面：", expected_pages)
print("找到的必要页面：", hit_pages)
print(f"必要页面覆盖率：{hit_count}/{len(expected_pages)} = {coverage:.2f}")
print("第一条资料摘要：", baseline[0].text[:180], "...")

assert coverage < 1.0

返回页面： [44, 131, 126, 139]
检索后用于核对的必要页面： [41, 42, 43, 44]
找到的必要页面： [44]
必要页面覆盖率：1/4 = 0.25
第一条资料摘要： 由于存在约束tr(WTSwW) = N−1 P i=1 wT i Swwi = 1，所以欲使上式取到最大值，只需取N −1 个最大的λi 即 可。根据Sbwi = λiSwwi 可知，λi 对应的便是广义特征值，wi 是λi 所对应的特征向量。 （广义特征值的定义和常用求解方法可查阅[3]） 对于N 分类问题，一定要求出N −1 个wi 吗？其实不然。之所以 ...


## 接下来检查什么

第一条结果是推导的最后一页，但其余结果来自其他章节，四个必要页面只找到一个。问题出在资料没有找全，不是回答措辞。后面的[找到相关页面后补上相邻内容](../6.%20处理信息缺口/找到相关页面后补上相邻内容.ipynb)会处理它。

教程中的其他方法也遵循同一要求：先保存原始结果，再只改一个环节，再比较同一个问题的实际变化。

In [4]:
print("本页只保存改动前确实答不好的结果，不把它当作任何方法的效果证明。")

本页只保存改动前确实答不好的结果，不把它当作任何方法的效果证明。




## 评估工作流中的第一步

先保存原问题、必要证据、修改前的检索结果、排名和运行参数，再选择一个改法。本页用线性判别分析的连续推导问题显示资料缺口；比较前后不能更换问题、资料版本或返回数量。
